# Bases de données — Exercices 4 et 5 : clés primaires (correction)

Ce notebook contient, pour chaque question, trois blocs **toujours séparés** :

| Bloc | Contenu |
|:---|:---|
| 📝 **ÉNONCÉ** | la question, recopiée du sujet |
| ✅ **RÉSOLUTION** | la réponse attendue (requête SQL, code et/ou texte) |
| 💡 **EXPLICATION PÉDAGOGIQUE** | pourquoi ça marche, le lien avec le cours, les erreurs à éviter. **N'est pas à recopier.** |

**Notions travaillées :** identifiant (clé), clés candidates, clé primaire, unicité des enregistrements, contrainte de clé primaire.


Exécuter les cellules **dans l'ordre** (*Cell → Run All*).

> **Rappel du cours (définition).** Un **identifiant** (appelé aussi **clé**) d'une relation est la donnée d'un ou plusieurs attributs qui permet d'identifier de manière unique un enregistrement. En d'autres termes, deux enregistrements ne peuvent avoir le même identifiant.
> Dans une relation, il y a en général plusieurs choix possibles pour un identifiant (appelés **clés candidates**). La donnée d'un identifiant d'une relation est appelée **clé primaire**.

---
# MISE EN PLACE

En classe, les requêtes sont tapées dans la **console H2** (un SGBD). Ici, on utilise le module `sqlite3` de Python, qui contient un autre SGBD (**SQLite**) : **le langage SQL est le même**, donc chaque requête de ce notebook peut être recopiée telle quelle dans la console H2 (les rares différences sont signalées).

- `execute_fichier("fichier.sql")` joue le rôle de `RUNSCRIPT FROM 'fichier.sql';` dans H2 ;
- `affiche(requete)` exécute une requête et affiche le résultat sous forme de tableau, comme la console H2 ;
- `resultat(requete)` renvoie la liste des enregistrements : elle ne sert qu'aux vérifications.

La requête SQL est écrite entre trois guillemets `""" ... """` : c'est une chaîne de caractères Python qui peut tenir sur plusieurs lignes.

Ces exercices sont des questions de réflexion : le code ne sert qu'à **illustrer** les réponses en montrant comment le SGBD réagit.

In [1]:
import sqlite3

base = sqlite3.connect(":memory:")   # une base de données vide, en mémoire vive


def execute_fichier(nom_fichier):
    """Équivalent de RUNSCRIPT FROM '...' dans H2 : exécute un fichier .sql."""
    fichier = open(nom_fichier, encoding="utf-8")
    script = fichier.read()
    fichier.close()
    base.executescript(script)


def resultat(requete):
    """Renvoie la liste des enregistrements obtenus (pour les vérifications)."""
    return base.execute(requete).fetchall()


def affiche(requete, maximum=20):
    """Exécute une requête et affiche le résultat comme dans la console H2."""
    curseur = base.execute(requete)
    if curseur.description is None:          # INSERT, UPDATE, DELETE, CREATE...
        print("Instruction exécutée.")
        return
    entete = ""
    for colonne in curseur.description:
        entete = entete + colonne[0] + " | "
    print(entete)
    nombre = 0
    for ligne in curseur:
        if nombre < maximum:
            texte = ""
            for valeur in ligne:
                texte = texte + str(valeur) + " | "
            print(texte)
        nombre = nombre + 1
    if nombre > maximum:
        print("... (seules les " + str(maximum) + " premières lignes sont affichées)")
    print("(" + str(nombre) + " ligne(s))")

---
# EXERCICE 4

#### 📝 ÉNONCÉ

> On considère une relation regroupant des informations sur les élèves d'un lycée, dont les attributs sont les suivants : nom, prénom, date de naissance, adresse, matricule.
> Parmi les données ci-dessous, lesquelles peuvent être utilisées comme clé primaire ? Justifier.
>
> - nom ;
> - adresse ;
> - (nom, prénom) ;
> - (nom, prénom, adresse) ;
> - matricule ;
> - (nom, matricule).

#### ✅ RÉSOLUTION

La question à se poser pour chaque proposition : **peut-il exister deux élèves différents avec la même valeur ?** Si oui, ce n'est pas une clé.

| Proposition | Clé primaire ? | Justification |
|:---|:---:|:---|
| nom | ❌ | Deux élèves peuvent avoir le même nom (deux frères et sœurs, deux « Martin »). |
| adresse | ❌ | Des frères et sœurs, ou des élèves d'un même immeuble, partagent la même adresse. |
| (nom, prénom) | ❌ | Les homonymes existent : deux « Léa Martin » peuvent être dans le même lycée. |
| (nom, prénom, adresse) | ❌ | C'est rare, mais **pas garanti** : par exemple deux cousins homonymes hébergés à la même adresse. De plus, une adresse change quand on déménage, alors qu'un identifiant doit rester stable. |
| **matricule** | ✅ | Le matricule est **attribué** par l'établissement justement pour être **unique** : deux élèves n'ont jamais le même. C'est **le bon choix** de clé primaire. |
| (nom, matricule) | ✅ (mais à éviter) | Deux élèves ne peuvent pas avoir le même couple (nom, matricule), puisque le matricule suffit déjà à les distinguer. Mais le nom est **inutile** : on choisit une clé **minimale**, donc `matricule` seul. |

**Réponse :** peuvent servir de clé primaire `matricule` et `(nom, matricule)` ; on choisit `matricule`, car il suffit à lui seul.

*Illustration (non demandée) : deux élèves homonymes (données inventées).*

In [2]:
# Avec (nom, prénom) comme clé primaire, deux homonymes sont refusés par le SGBD
affiche("""
CREATE TABLE Eleves1 (
    nom VARCHAR, prenom VARCHAR, naissance DATE, adresse VARCHAR, matricule INT,
    PRIMARY KEY (nom, prenom) );
""")
affiche("INSERT INTO Eleves1 VALUES ('Martin', 'Léa', '2008-03-12', '3 rue Hoche', 1001)")
try:
    affiche("INSERT INTO Eleves1 VALUES ('Martin', 'Léa', '2009-07-01', '8 avenue Foch', 1002)")
except sqlite3.IntegrityError as erreur:
    print("Refusé par le SGBD :", erreur)

# Avec matricule comme clé primaire, les deux élèves sont acceptés
affiche("""
CREATE TABLE Eleves2 (
    nom VARCHAR, prenom VARCHAR, naissance DATE, adresse VARCHAR, matricule INT,
    PRIMARY KEY (matricule) );
""")
affiche("INSERT INTO Eleves2 VALUES ('Martin', 'Léa', '2008-03-12', '3 rue Hoche', 1001)")
affiche("INSERT INTO Eleves2 VALUES ('Martin', 'Léa', '2009-07-01', '8 avenue Foch', 1002)")
affiche("SELECT * FROM Eleves2")

Instruction exécutée.
Instruction exécutée.
Refusé par le SGBD : UNIQUE constraint failed: Eleves1.nom, Eleves1.prenom
Instruction exécutée.
Instruction exécutée.
Instruction exécutée.
nom | prenom | naissance | adresse | matricule | 
Martin | Léa | 2008-03-12 | 3 rue Hoche | 1001 | 
Martin | Léa | 2009-07-01 | 8 avenue Foch | 1002 | 
(2 ligne(s))


#### 💡 EXPLICATION PÉDAGOGIQUE

- **Le test d'une clé :** une clé doit être unique **pour toutes les données possibles**, pas seulement pour les données actuelles. « Il n'y a pas d'homonymes cette année » ne suffit pas : il pourrait y en avoir l'an prochain.
- **Pourquoi des identifiants artificiels ?** Dans la vie réelle, presque aucune caractéristique naturelle n'est unique (noms, adresses, dates de naissance). C'est pour cela qu'on **invente** des numéros : matricule, numéro de sécurité sociale, numéro de commande, `id` dans les tables de l'exercice 2...
- **Clés candidates :** `matricule` et `(nom, matricule)` sont toutes deux des identifiants. On choisit comme **clé primaire** le plus simple (minimal).
- **Ce que fait le SGBD :** une fois la clé primaire déclarée, il **refuse** toute insertion qui créerait un doublon de clé (c'est la *contrainte de clé primaire*). Dans l'illustration, avec la clé `(nom, prénom)`, la deuxième Léa Martin ne peut pas être inscrite : c'est bien la preuve que ce choix de clé est mauvais.
- `try ... except` sert seulement à afficher le message d'erreur sans arrêter le notebook.

---
# EXERCICE 5

#### 📝 ÉNONCÉ

> Une relation contient des informations sur les trajets réalisés par les utilisateurs d'un service de location de vélos. Les attributs de cette relation sont les suivants : date, heures de départ et d'arrivée, stations de départ et d'arrivée.
> L'unicité des enregistrements est-elle garantie ? Proposer une solution afin de pouvoir définir une clé primaire.

#### ✅ RÉSOLUTION

**L'unicité n'est pas garantie.** Deux utilisateurs différents peuvent emprunter un vélo **à la même station, à la même heure**, et le rendre **à la même station, à la même heure** (deux amis qui font le trajet ensemble, par exemple). On aurait alors deux enregistrements identiques : aucun attribut, ni aucun groupe d'attributs, ne permet de les distinguer.

**Solution :** ajouter un attribut qui sert d'identifiant, par exemple :
- un **numéro de trajet** `id_trajet`, attribué par le service et différent pour chaque trajet (comme l'attribut `id` des tables de l'exercice 2). Il devient la clé primaire :

  Trajets(<u>id_trajet</u>, date, heure_depart, heure_arrivee, station_depart, station_arrivee)

- ou bien un **identifiant de l'utilisateur** `id_utilisateur` : comme un utilisateur ne peut pas commencer deux trajets au même instant, le groupe (`id_utilisateur`, `date`, `heure_depart`) identifie un trajet. Cela suppose cependant qu'un utilisateur ne loue qu'un vélo à la fois.

La première solution est la plus simple et la plus sûre.

*Illustration (non demandée) : deux amis font le même trajet (données inventées). Grâce à `id_trajet`, les deux trajets sont distincts.*

In [3]:
affiche("""
CREATE TABLE Trajets (
    id_trajet INT, date_trajet DATE, heure_depart TIME, heure_arrivee TIME,
    station_depart VARCHAR, station_arrivee VARCHAR,
    PRIMARY KEY (id_trajet) );
""")
affiche("INSERT INTO Trajets VALUES (1, '2026-11-14', '08:05', '08:20', 'Lecourbe', 'Bastille')")
affiche("INSERT INTO Trajets VALUES (2, '2026-11-14', '08:05', '08:20', 'Lecourbe', 'Bastille')")
affiche("SELECT * FROM Trajets")

Instruction exécutée.
Instruction exécutée.
Instruction exécutée.
id_trajet | date_trajet | heure_depart | heure_arrivee | station_depart | station_arrivee | 
1 | 2026-11-14 | 08:05 | 08:20 | Lecourbe | Bastille | 
2 | 2026-11-14 | 08:05 | 08:20 | Lecourbe | Bastille | 
(2 ligne(s))


#### 💡 EXPLICATION PÉDAGOGIQUE

- **Méthode :** pour montrer que l'unicité n'est **pas** garantie, il suffit de trouver **un exemple concret** de deux enregistrements différents (deux vrais trajets) qui auraient exactement les mêmes valeurs. Un seul contre-exemple suffit.
- **Rédaction :** une bonne réponse donne le contre-exemple **et** propose une solution précise (le nom de l'attribut ajouté, et le schéma de la relation avec la clé soulignée).
- Dans l'illustration, les deux enregistrements ne diffèrent **que** par `id_trajet` : sans cet attribut, ce seraient deux lignes identiques, impossibles à distinguer (et à modifier ou supprimer séparément).
- Les attributs ont été renommés `date_trajet`, `heure_depart`... : `DATE` et `TIME` sont des **mots réservés** de SQL (des noms de types), à éviter comme noms d'attributs.